In [30]:
import os
import json
import shutil
import subprocess
import numpy as np
import yaml
from mendeleev import element

In [ ]:
MATERIAL_LIBRARY = "../mineral_library.yaml"
MATERIALS_MACRO  = os.path.abspath("./build/materials.mac")

# Materials not in the YAML library (same keys: nuclei, stoich, density_g_cm3)
EXTRA_MATERIALS = {
    "Diamond": {"nuclei": ["C"], "stoich": [1], "density_g_cm3": 3.5},
}

def write_materials_macro(yaml_path=MATERIAL_LIBRARY, out_path=MATERIALS_MACRO, extra=EXTRA_MATERIALS):
    """
    Writes a Geant4 macro with one /testhadr/defineMaterial command per material.
    stoich = relative atom counts; Geant4 converts them to mass fractions.
    """
    with open(yaml_path) as f:
        library = yaml.safe_load(f)
    library.update(extra or {})

    lines = [f"# Auto-generated from {yaml_path} -- do not edit by hand"]
    for key, m in library.items():
        if len(m["nuclei"]) != len(m["stoich"]):
            raise ValueError(f"{key}: nuclei and stoich have different lengths")
        comps = " ".join(f"{el} {n}" for el, n in zip(m["nuclei"], m["stoich"]))
        lines.append(f"/testhadr/defineMaterial {key} {m['density_g_cm3']} {comps}")

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"Wrote {len(library)} material definitions to {out_path}")
    return sorted(library)

# Densities of the materials hardcoded in DetectorConstruction.cc (not in the YAML)
HARDCODED_DENSITIES_G_CM3 = {"StdRock": 2.65, "H2O": 1.0}

def material_density_g_cm3(mineral_name, yaml_path=MATERIAL_LIBRARY, extra=EXTRA_MATERIALS):
    """Density used for mineral_name in Geant4, or None if unknown."""
    if mineral_name in HARDCODED_DENSITIES_G_CM3:
        return HARDCODED_DENSITIES_G_CM3[mineral_name]
    with open(yaml_path) as f:
        library = yaml.safe_load(f)
    library.update(extra or {})
    entry = library.get(mineral_name)
    return float(entry["density_g_cm3"]) if entry else None

write_materials_macro()

FileNotFoundError: [Errno 2] No such file or directory: './mineral_library.yaml'

In [32]:
# Target length per material [m]; anything not listed uses DEFAULT_TARGET_LENGTH_M.
# StdRock is longer because its runs calibrate the muon penetration depth.
DEFAULT_TARGET_LENGTH_M = 200.
TARGET_LENGTH_M = {"StdRock": 1000.}

def run_geant4_simulation(mineral_name, particle_type, n_events, energies_gev, ion_type=None,
                          materials_macro=MATERIALS_MACRO, target_length_m=None, target_radius_m=10.):
    """
    Dynamically creates a Geant4 macro file and runs the simulation for a given list of energies.

    Args:
        mineral_name (str): Name of the target material (must be defined in Geant4).
        particle_type (str): Particle type for the gun (e.g., "mu-", "proton").
        n_events (int): Number of primary particles to simulate per energy.
        energies_gev (np.array): Array of energies in GeV to simulate.
        materials_macro (str): Macro with /testhadr/defineMaterial commands (see write_materials_macro).
        target_length_m (float): Length of the target cylinder in m. If None, taken from
            TARGET_LENGTH_M (falls back to DEFAULT_TARGET_LENGTH_M).
        target_radius_m (float): Radius of the target cylinder in m.
    """

    if target_length_m is None:
        target_length_m = TARGET_LENGTH_M.get(mineral_name, DEFAULT_TARGET_LENGTH_M)
    if target_length_m <= 0 or target_radius_m <= 0:
        raise ValueError("target_length_m and target_radius_m must be > 0")

    default_output_dir = "./build/output"
    final_output_dir = f"./{mineral_name}/{particle_type}"
    
    if not os.path.exists(default_output_dir):
        os.makedirs(default_output_dir)
    
    if not os.path.exists(final_output_dir):
        os.makedirs(final_output_dir)


    macro_content = f"""# Auto-generated macro file
/control/verbose 0
/run/verbose 1
/tracking/verbose 0

# --- Materials ---
/control/execute           {materials_macro}

# --- Geometry Setup ---
/testhadr/TargetMat        {mineral_name}
/testhadr/TargetRadius     {target_radius_m} m
/testhadr/TargetLength     {target_length_m} m

# --- Physics ---
/run/setCut                0.005 mm
/run/initialize

# --- Run Loop ---
/gun/particle {particle_type}
"""

    if particle_type=="ion":
        macro_content +=f"""
/gun/ion {ion_type[0]} {ion_type[1]} {ion_type[0]} 0.0
"""

    for energy in energies_gev:
        macro_content += f"\n/gun/energy {energy:.16f} GeV\n"
        macro_content += f"/run/beamOn {n_events}\n"

    macro_filename = "./build/run.in"
    with open(macro_filename, "w") as f:
        f.write(macro_content)

    executable = "./build/Run"
    
    if not os.path.exists(executable):
        print(f"Error: Executable '{executable}' not found. Compile the C++ code first.")
        return

    print(f"Starting Geant4 simulation for {mineral_name} with {len(energies_gev)} energy points...")
    
    try:
        result = subprocess.run([executable, macro_filename], check=True, capture_output=True, text=True)
        print(result.stdout)
        
    except subprocess.CalledProcessError as e:
        print("Error running Geant4:")
        print(e.stderr)
        return
    
    print(f"Simulation finished. Moving files to {final_output_dir}...")
    
    for i, energy in enumerate(energies_gev):
        src_filename = f"outNuclei_{i}.txt"
        src_path = os.path.join(default_output_dir, src_filename)
        
        dst_filename = f"outNuclei_{energy:.6f}.txt"
        dst_path = os.path.join(final_output_dir, dst_filename)
        
        if os.path.exists(src_path):
            shutil.move(src_path, dst_path)
        else:
            print(f"Warning: Expected output file {src_filename} not found.")

    # Geometry sidecar read by mineral_utils: the gun sits on the target top
    # face, so depth below it = depth_reference_z_m - z.
    geometry = {
        "target_length_m": float(target_length_m),
        "target_radius_m": float(target_radius_m),
        "depth_reference_z_m": float(target_length_m) / 2.,
        "density_g_cm3": material_density_g_cm3(mineral_name),
    }
    with open(os.path.join(final_output_dir, "geometry.json"), "w") as f:
        json.dump(geometry, f, indent=2)

    print("All done.")




In [33]:
def get_all_known_isotopes(max_z=65):
    """
    Returns (Z, A) for every bound isotope documented in mendeleev up to max_z.
    """
    isotopes_list = []
    
    for z in range(1, max_z + 1):
        el = element(z)
        for iso in el.isotopes:
            isotopes_list.append((z, iso.mass_number))
            
    return list(set(isotopes_list))

def export_all_stopping_powers(mineral_name, max_z=65, materials_macro=MATERIALS_MACRO):
    """
    Runs the Geant4 executable in bypass mode to export dE/dx tables.
    """
    output_dir = f"./{mineral_name}/Range"
    os.makedirs(output_dir, exist_ok=True)
    
    isotopes = get_all_known_isotopes(max_z)
    
    executable = "./build/Run"
    if not os.path.exists(executable):
        print(f"Error: Executable '{executable}' not found.")
        return
        
    print(f"Exporting tables for {len(isotopes)} isotopes in {mineral_name}...")
    
    # 2. Loop and call the bypass command
    for Z, A in isotopes:
        try:
            subprocess.run(
                [executable, "--export-dedx", mineral_name, str(Z), str(A), output_dir, materials_macro],
                check=True,
                capture_output=True, # Keeps the terminal clean
                text=True
            )
        except subprocess.CalledProcessError as e:
            print(f"Error exporting Z={Z}, A={A}:")
            print(e.stderr)

    print(f"Done! All tables saved to {output_dir}")



In [34]:
energies = np.logspace(-3, 4, 120)[:90]

In [36]:
run_geant4_simulation("StdRock", "mu-", 10000, energies)

Starting Geant4 simulation for StdRock with 90 energy points...

**************************************************************
 Geant4 version Name: geant4-11-03-patch-02 [MT]   (25-April-2025)
                       Copyright : Geant4 Collaboration
                      References : NIM A 506 (2003), 250-303
                                 : IEEE-TNS 53 (2006), 270-278
                                 : NIM A 835 (2016), 186-225
                             WWW : http://geant4.org/
**************************************************************

### Defined material Halite, density 2.16 g/cm3, mass fractions: Na=0.3934 Cl=0.6066
### Defined material Olivine, density 3.3 g/cm3, mass fractions: Mg=0.2972 Fe=0.07705 Si=0.1911 O=0.4347
### Defined material Quartz, density 2.6 g/cm3, mass fractions: Si=0.4674 O=0.5326
### Defined material Spinel, density 3.7 g/cm3, mass fractions: Mg=0.1708 Al=0.3793 O=0.4498
### Defined material Apatite, density 3.15 g/cm3, mass fractions: Ca=0.3971 P=0.

In [ ]:
run_geant4_simulation("Halite", "mu-", 10000, energies)
run_geant4_simulation("Halite", "mu+", 10000, energies)
#export_all_stopping_powers("Halite", max_z=65)

Starting Geant4 simulation for Halite with 90 energy points...


In [ ]:
run_geant4_simulation("Olivine", "mu-", 10000, energies)
run_geant4_simulation("Olivine", "mu+", 10000, energies)
#export_all_stopping_powers("Olivine", max_z=65)

In [ ]:
run_geant4_simulation("Diamond", "mu-", 10000, energies)
run_geant4_simulation("Diamond", "mu+", 10000, energies)
#export_all_stopping_powers("Diamond", max_z=65)

In [ ]:
run_geant4_simulation("Halite", "neutron", 10000, energies)

In [ ]:
run_geant4_simulation("Olivine", "neutron", 10000, energies)

In [ ]:
run_geant4_simulation("Diamond", "neutron", 10000, energies)

In [ ]:
run_geant4_simulation("Quartz", "mu-", 10000, energies)
run_geant4_simulation("Quartz", "mu+", 10000, energies)
export_all_stopping_powers("Quartz", max_z=65)

In [ ]:
run_geant4_simulation("Quartz", "neutron", 10000, energies)